In [ ]:
from datetime import datetime
from getpass import getpass
import os

rdm_url = None
idp_name_1 = None
idp_username_1 = None
idp_password_1 = None
default_result_path = None
close_on_fail = False
project_name = None
transition_timeout = 60 * 1000

binderhub_base_image = '#repo2docker#python=3.12.*,r-base=4.4.*'

binderhub_apt_package = 'sl'
binderhub_conda_package = 'awscli'
binderhub_pip_package = 'papermill'
binderhub_r_package = 'eegkit'

binderhub_post_build_script = '''#!/bin/bash
date > ~/uptime'''
binderhub_binder_files = ['apt.txt', 'environment.yml', 'install.R', 'postBuild']
binderhub_launch_timeout = 30 * 60 * 1000   # 30 minutes
binderhub_test_filename = 'grdm_test_file.txt'
binderhub_test_file_content = 'GRDM_FILE_SYNC_TEST'

In [ ]:
if rdm_url is None:
    rdm_url = input(prompt=f'RDM URL: ')
if idp_name_1 is None:
    idp_name_1 = input(prompt=f'IdP Name: ')
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if project_name is None:
    project_name = datetime.now().strftime('TEST-BINDERHUB-%Y%m%d%H%M')

project_url = None
project_created = False


# BinderHubアドオン repo2docker による解析環境の起動

- サブシステム名: アドオン
- ページ/アドオン: BinderHub
- 機能分類: アドオン操作
- シナリオ名: repo2docker基本イメージを使った解析環境の起動
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM, BinderHub, GRDMは全てプロフィールを埋めていること / JupyterHubはサーバーが5つ以内の状態であること)、BinderHub OAuthクライアント情報
- 事前条件: 「プロジェクトに対するBinderHubアドオンの登録」を実施済みであること


In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir


In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)


In [ ]:
import asyncio

async def run_lab_command(page, command, expected_substring, run_wait_time=3000):
    editor = page.locator('div.jp-CodeCell div.cm-content').first
    await expect(editor).to_be_visible(timeout=transition_timeout)
    await editor.click()
    await editor.fill(command)

    run_button = page.locator('//jp-button[@data-command = "notebook:run-cell-and-select-next"]')
    await expect(run_button).to_be_visible(timeout=transition_timeout)
    await run_button.click()
    await asyncio.sleep(run_wait_time / 1000)

    output = page.locator(f'//*[contains(@class, "jp-OutputArea")]//*[text()[contains(., "{expected_substring}")]]')
    await expect(output).to_be_visible(timeout=transition_timeout)


## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url, wait_until="networkidle")
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()

await run_pw(_step)


## IdPを利用し、既存ユーザー1としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)


## ダッシュボードから「新規プロジェクト作成」をクリックする

指定したプロジェクトが存在しない場合、新規プロジェクトが作成されること

In [ ]:
async def _step(page):
    global project_created
    project_created = await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)
    if project_created:
        print(f'Created project: {project_name}')
    else:
        print(f'Project already exists: {project_name}')

await run_pw(_step)


## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    global project_url
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)


## BinderHubアドオンを有効化する

アドオン利用規約の確認ダイアログが表示されること

In [ ]:
async def _step(page):
    await grdm.enable_addon(page, 'GakuNin Federated Computing Services (Jupyter)', transition_timeout=transition_timeout)

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「解析」をクリックする

BinderHubアドオンページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "解析")]').click()
    open_idp_list = page.locator('//*[@id = "dropdown_img"]')
    launch_button = page.locator('//*[@data-test-binderhub-launch]')
    await expect(open_idp_list.or_(launch_button)).to_be_visible(timeout=transition_timeout)

    if await open_idp_list.is_visible():
        await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)

    await expect(launch_button).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 基本イメージの「変更」をクリックする

基本イメージ一覧が表示されること

In [ ]:
async def _step(page):
    addon_name = 'GakuNin Federated Computing Services (Jupyter)'
    enable_locator = page.locator(f'//div[@full_name = "{addon_name}"]//a[text() = "有効にする"]')
    if await enable_locator.count():
        await enable_locator.click()
        confirm_button = page.locator('//button[@data-bb-handler = "confirm"]')
        await expect(confirm_button).to_be_visible(timeout=transition_timeout)
        await confirm_button.click()
    else:
        print('Addon already enabled')

await run_pw(_step)
async def _step(page):
    await page.locator('//*[@data-test-image-change]').click()
    await expect(page.locator(f'//*[@data-test-image-selection = "{binderhub_base_image}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「Python 3.12 + R 4.4 (JupyterLab 4.x)」をクリックする

「追加パッケージ」エディターが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-image-selection = "{binderhub_base_image}"]').click()
    await expect(page.locator('//*[@data-test-package-add]')).not_to_have_count(0, timeout=transition_timeout)
    await expect(page.locator('//div[@data-test-package-editor = "apt"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//div[@data-test-package-editor = "conda"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//div[@data-test-package-editor = "pip"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//div[@data-test-package-editor = "rmran"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「apt-get」の「+追加」をクリックし、パッケージを登録する

パッケージ名がaptに表示されること

In [ ]:
async def _step(page):
    await page.locator('//div[@data-test-package-editor = "apt"]//*[@data-test-package-add]').click()
    field = page.locator('//input[@name = "package_name"]')
    await expect(field).to_be_visible(timeout=transition_timeout)
    await field.fill(binderhub_apt_package)
    await page.locator('//button[@data-test-package-item-confirm]').click()
    await expect(page.locator(f'//div[@data-test-package-editor = "apt"]//*[text() = "{binderhub_apt_package}"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(transition_timeout / 1000)

await run_pw(_step)

## 「conda」の「+追加」をクリックし、パッケージを登録する

パッケージ名がcondaに表示されること

In [ ]:
async def _step(page):
    await page.locator('//div[@data-test-package-editor = "conda"]//*[@data-test-package-add]').click()
    field = page.locator('//input[@name = "package_name"]')
    await expect(field).to_be_visible(timeout=transition_timeout)
    await field.fill(binderhub_conda_package)
    await page.locator('//button[@data-test-package-item-confirm]').click()
    await expect(page.locator(f'//div[@data-test-package-editor = "conda"]//*[text() = "{binderhub_conda_package}"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(transition_timeout / 1000)

await run_pw(_step)

## 「pip」の「+追加」をクリックし、パッケージを登録する

パッケージ名がpipに表示されること

In [ ]:
async def _step(page):
    await page.locator('//div[@data-test-package-editor = "pip"]//*[@data-test-package-add]').click()
    field = page.locator('//input[@name = "package_name"]')
    await expect(field).to_be_visible(timeout=transition_timeout)
    await field.fill(binderhub_pip_package)
    await page.locator('//button[@data-test-package-item-confirm]').click()
    await expect(page.locator(f'//div[@data-test-package-editor = "pip"]//*[text() = "{binderhub_pip_package}"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(transition_timeout / 1000)

await run_pw(_step)

## 「R (CRAN)」の「+追加」をクリックし、パッケージを登録する

パッケージ名がR (CRAN)に表示されること

In [ ]:
async def _step(page):
    await page.locator('//div[@data-test-package-editor = "rmran"]//*[@data-test-package-add]').click()
    field = page.locator('//input[@name = "package_name"]')
    await expect(field).to_be_visible(timeout=transition_timeout)
    await field.fill(binderhub_r_package)
    await page.locator('//button[@data-test-package-item-confirm]').click()
    await expect(page.locator(f'//div[@data-test-package-editor = "rmran"]//*[text() = "{binderhub_r_package}"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(transition_timeout / 1000)

await run_pw(_step)

## 「自動実行スクリプト」をクリックする

スクリプトエディタが表示されること

In [ ]:
async def _step(page):
    await page.locator('//h3//button/*[contains(@class, "fa-chevron-right")]').click()
    await expect(page.locator('//textarea[@data-test-post-build]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//button[@data-test-save-post-build]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## スクリプトを入力し、保存する

保存ボタンが有効になり、保存後に無効化されること

In [ ]:
async def _step(page):
    textarea = page.locator('//textarea[@data-test-post-build]')
    await textarea.fill(binderhub_post_build_script)
    await textarea.press('Tab')
    save_button = page.locator('//button[@data-test-save-post-build]')
    await expect(save_button).to_be_enabled(timeout=transition_timeout)
    await save_button.click()
    await expect(save_button).to_be_disabled(timeout=transition_timeout)
    await page.wait_for_load_state('networkidle')
    await asyncio.sleep(transition_timeout / 1000)

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「ファイル」をクリックする

フォルダ「.binder」が作成されていること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-node-navbar-link = "files"]').click()
    binder_folder = page.locator('//*[text() = ".binder"]/../..//*[contains(@class, "fa-plus")]')
    await expect(binder_folder).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「.binder」フォルダを開き、生成されたファイルを確認する

apt.txt などのファイルが表示されること

In [ ]:
async def _step(page):
    toggle = page.locator('//*[text() = ".binder"]/../..//*[contains(@class, "fa-plus")]')
    await toggle.click()
    for filename in binderhub_binder_files:
        await expect(page.locator(f'//*[text() = "{filename}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## プロジェクトのストレージにテストファイルをアップロードする

ファイルがアップロードされ、ファイル一覧に表示されること

In [ ]:
async def _step(page):
    # テストファイルを作成
    test_filepath = os.path.join(work_dir, binderhub_test_filename)
    with open(test_filepath, 'w') as f:
        f.write(binderhub_test_file_content)

    # NII Storageを選択してアップロード
    await grdm.get_select_storage_title_locator(page, 'NII Storage').click()
    await grdm.upload_file(page, test_filepath)
    await expect(grdm.get_select_file_title_locator(page, binderhub_test_filename)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「解析」をクリックする

BinderHubアドオンページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "解析")]').click()
    await expect(page.locator('//*[@data-test-binderhub-launch]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「新しい解析環境を作成」をクリックする

環境の起動を待つ

In [ ]:
async def _step(page):
    popup_future = page.wait_for_event('popup', timeout=binderhub_launch_timeout)
    await page.locator('//*[@data-test-binderhub-launch]').click()
    popup = await popup_future
    return popup

await run_pw(_step)


## JupyterLabが表示されていることを確認する

In [ ]:
async def _step(page):
    open_idp_list = page.locator('//*[@id = "dropdown_img"]')
    start_ipykernel = page.locator(f'//*[@class = "jp-LauncherCard" and @title = "Python 3 (ipykernel)" and @data-category = "Notebook"]')
    await expect(open_idp_list.or_(start_ipykernel)).to_be_visible(timeout=transition_timeout)

    if await open_idp_list.is_visible():
        await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)

    await expect(start_ipykernel).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 新規Notebookを作成する

In [ ]:
async def _step(page):
    launcher_label = 'Python 3 (ipykernel)'
    launcher = page.locator(f'//*[@class = "jp-LauncherCard" and @title = "{launcher_label}" and @data-category = "Notebook"]')
    await expect(launcher).to_be_visible(timeout=transition_timeout)
    await launcher.click()

    await expect(page.locator('//*[contains(@class, "jp-Notebook-ExecutionIndicator") and @data-status="idle"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「!which /usr/games/sl」を実行して結果を確認する

コマンドの出力に `/usr/games/sl` が含まれること

In [ ]:
async def _step(page):
    await run_lab_command(page, '!which /usr/games/sl', '/usr/games/sl')

await run_pw(_step)


## 「!aws」を実行して結果を確認する

コマンドの出力に `usage: aws` が含まれること

In [ ]:
async def _step(page):
    await run_lab_command(page, '!aws', 'usage: aws')

await run_pw(_step)


## 「!papermill」を実行して結果を確認する

コマンドの出力に `Usage: papermill` が含まれること

In [ ]:
async def _step(page):
    await run_lab_command(page, '!papermill', 'Usage: papermill')

await run_pw(_step)


## GRDMプロジェクトのファイルがコピーされていることを確認する

テストファイルの内容が出力されること

In [ ]:
async def _step(page):
    await run_lab_command(page, f'!cat {binderhub_test_filename}', binderhub_test_file_content)

await run_pw(_step)


## Notebookを閉じる

In [ ]:
async def _step(page):
    close_icon = page.locator('//*[contains(@class, "lm-mod-current")]//*[contains(@class, "lm-TabBar-tabCloseIcon")]')
    await expect(close_icon).to_be_visible(timeout=transition_timeout)
    await close_icon.click()

    discard_button = page.locator('//div[text() = "Discard"]')
    await expect(discard_button).to_be_visible(timeout=transition_timeout)
    await discard_button.click()

await run_pw(_step)

## 「R」ランチャーを起動する

In [ ]:
async def _step(page):
    launcher_label = 'R'
    launcher = page.locator(f'//*[@class = "jp-LauncherCard" and @title = "{launcher_label}" and @data-category = "Notebook"]')
    await expect(launcher).to_be_visible(timeout=transition_timeout)
    await launcher.click()
    await expect(page.locator('div.jp-CodeCell div.cm-content').first).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「R」ランチャーで `library(eegkit)` を実行する

出力に `Loading required package: eegkitdata` が含まれること

In [ ]:
async def _step(page):
    await expect(page.locator('//*[contains(@class, "jp-Notebook-ExecutionIndicator") and @data-status="idle"]')).to_be_visible(timeout=10 * 1000)
    await run_lab_command(page, 'library(eegkit)', 'Loading required package: eegkitdata')

await run_pw(_step)


## JupyterLabウィンドウを閉じる

In [ ]:
await close_latest_page()

async def _step(page):
    await expect(page.locator('//*[@data-test-binderhub-launch]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


終了処理を実施する。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}